<a href="https://colab.research.google.com/github/miriamamin1213-ux/AUC_Curves/blob/main/HecktorGPT2LoRASHAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# HECKTOR GPT-2 LoRA
# HPV Status Prediction | Compact Single-Cell Notebook
# ============================================================

# 1. IMPORT LIBRARIES, SET SEEDS AND SELECT DEVICE
import os, random, gdown
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, balanced_accuracy_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve
)
import torch
import transformers
import peft

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

!pip uninstall -y torchao


SEED = 42
def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# 2. LOAD HECKTOR EXCEL DATA
DATA_FILE = "HPV2025.xlsx"
if not os.path.exists(DATA_FILE):
    downloaded = gdown.download(
        id="1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv",
        output=DATA_FILE, quiet=False
    )
    if downloaded is None or not os.path.exists(DATA_FILE):
        raise FileNotFoundError("Could not download HPV2025.xlsx.")

df = pd.read_excel(DATA_FILE)
TARGET = "HPV Status"
FEATURES = [
    "Age", "Gender", "Tobacco Consumption", "Alcohol Consumption",
    "Performance Status", "T-stage", "N-stage", "M-stage"
]
missing = [c for c in [TARGET] + FEATURES if c not in df.columns]
if missing:
    raise KeyError(f"Required Excel columns missing: {missing}")

# 3. ROBUSTLY NORMALISE TARGET AND SELECT FEATURES
def normalize_target(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in {"0", "0.0", "negative", "neg", "false", "no"}:
        return 0
    if s in {"1", "1.0", "positive", "pos", "true", "yes"}:
        return 1
    try:
        n = float(s)
        if n in (0, 1):
            return int(n)
    except (ValueError, TypeError):
        pass
    return np.nan

df["HPV_Label"] = df[TARGET].apply(normalize_target)
invalid_count = int(df["HPV_Label"].isna().sum())
df = df.dropna(subset=["HPV_Label"]).copy()
df["HPV_Label"] = df["HPV_Label"].astype(int)
df = df[FEATURES + ["HPV_Label"]].copy()

if df.empty:
    raise ValueError("No recognised HPV labels found; check the target column.")
counts = df["HPV_Label"].value_counts().reindex([0, 1], fill_value=0)
if (counts < 2).any():
    raise ValueError(f"Insufficient examples for both HPV classes: {counts.to_dict()}")

print("Dataset:", df.shape, "| Invalid/missing target rows dropped:", invalid_count)
print("HPV classes (0=negative, 1=positive):\n", counts)

# 4. STRATIFIED TRAIN / VALIDATION / TEST SPLIT (65/15/20)
train_val_df, test_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df["HPV_Label"]
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.1875, random_state=SEED,
    stratify=train_val_df["HPV_Label"]
)
train_df, val_df, test_df = [x.copy() for x in (train_df, val_df, test_df)]

# 5. IMPUTE MISSING FEATURES USING TRAINING DATA ONLY
fill_values = {}
for col in FEATURES:
    if pd.api.types.is_numeric_dtype(train_df[col]):
        value = train_df[col].median()
        if pd.isna(value):
            value = 0
    else:
        modes = train_df[col].dropna().astype(str).mode()
        value = modes.iloc[0] if not modes.empty else "Unknown"
    fill_values[col] = value
    for split in (train_df, val_df, test_df):
        split[col] = split[col].fillna(value)
        if split[col].dtype == object:
            split[col] = split[col].astype(str).replace(
                {"": "Unknown", "nan": "Unknown", "None": "Unknown"}
            )

y_train = train_df["HPV_Label"].to_numpy(dtype=int)
y_val = val_df["HPV_Label"].to_numpy(dtype=int)
y_test = test_df["HPV_Label"].to_numpy(dtype=int)
for name, y in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(f"{name}: n={len(y)}, class counts={np.bincount(y, minlength=2)}")

# 6. CONVERT CLINICAL FEATURES INTO TEXT PROMPTS
def display_value(x):
    if pd.isna(x):
        return "Unknown"
    if isinstance(x, (float, np.floating)) and float(x).is_integer():
        return str(int(x))
    return str(x).strip()

def format_stage(x, prefix):
    value = display_value(x)
    if value == "Unknown" or value.upper().startswith(prefix):
        return value
    try:
        n = float(value)
        if n.is_integer():
            return f"{prefix}{int(n)}"
    except ValueError:
        pass
    return f"{prefix}{value}"

def patient_to_text(row):

    age = display_value(row["Age"])
    gender = display_value(row["Gender"])
    tobacco = display_value(row["Tobacco Consumption"])
    alcohol = display_value(row["Alcohol Consumption"])
    performance = display_value(row["Performance Status"])

    t_stage = format_stage(row["T-stage"], "T")
    n_stage = format_stage(row["N-stage"], "N")
    m_stage = format_stage(row["M-stage"], "M")

    return (
        f"This patient is a {age}-year-old {gender}. "
        f"Tobacco consumption is {tobacco}. "
        f"Alcohol consumption is {alcohol}. "
        f"Performance status is {performance}. "
        f"Tumour stage is {t_stage}. "
        f"Nodal stage is {n_stage}. "
        f"Metastatic stage is {m_stage}. "
        "Predict whether the patient is HPV positive or HPV negative."
    )


train_texts = [patient_to_text(r) for _, r in train_df.iterrows()]
val_texts = [patient_to_text(r) for _, r in val_df.iterrows()]
test_texts = [patient_to_text(r) for _, r in test_df.iterrows()]

# 7. LOAD TOKENIZER AND SET TRAINING PARAMETERS
MODEL_NAME = "openai-community/gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
lengths = [len(tokenizer.encode(t, add_special_tokens=False)) for t in train_texts]
MAX_LENGTH = min(1024, max(32, int(np.ceil(max(lengths) / 8) * 8)))
BATCH_SIZE, EPOCHS = 64, 500
LEARNING_RATE, WEIGHT_DECAY, DROPOUT = 1e-3, 1e-4, 0.1
print("Max prompt length:", max(lengths), "| MAX_LENGTH:", MAX_LENGTH)
print("Batch size:", BATCH_SIZE, "| Epochs:", EPOCHS)

# 8. BUILD DATASETS AND DATA LOADERS
class HPVTextDataset(Dataset):
    def __init__(self, texts, labels):
        self.items = []
        for text, label in zip(texts, labels):
            enc = tokenizer(
                text, max_length=MAX_LENGTH, padding="max_length",
                truncation=True, return_tensors="pt"
            )
            self.items.append((
                enc["input_ids"].squeeze(0),
                enc["attention_mask"].squeeze(0), int(label)
            ))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        ids, mask, label = self.items[i]
        return {"input_ids": ids, "attention_mask": mask,
                "label": torch.tensor(label, dtype=torch.long)}

train_ds = HPVTextDataset(train_texts, y_train)
val_ds = HPVTextDataset(val_texts, y_val)
test_ds = HPVTextDataset(test_texts, y_test)
loader_args = {
    "batch_size": BATCH_SIZE, "num_workers": 0,
    "pin_memory": torch.cuda.is_available()
}
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, shuffle=True, generator=generator, **loader_args)
val_loader = DataLoader(val_ds, shuffle=False, **loader_args)
test_loader = DataLoader(test_ds, shuffle=False, **loader_args)


# ============================================================
# 9. INITIALISE GPT-2 CLASSIFIER AND CLASS-WEIGHTED LOSS
#     UPDATED: LoRA integration only
# ============================================================

from peft import LoraConfig, get_peft_model, TaskType


class HPVNetGPT2Text(nn.Module):
    def __init__(self):
        super().__init__()

        # Load the original GPT-2 backbone
        self.gpt2 = GPT2Model.from_pretrained(MODEL_NAME)
        self.gpt2.config.use_cache = False
        self.gpt2.config.pad_token_id = tokenizer.pad_token_id

        # Apply LoRA to GPT-2 attention projections
        lora_config = LoraConfig(
            r=6,
            lora_alpha=12,
            target_modules=["c_attn"],
            lora_dropout=0.0,
            bias="none",
            task_type=TaskType.FEATURE_EXTRACTION
        )

        self.gpt2 = get_peft_model(self.gpt2, lora_config)

        # Keep the original HECKTOR classification head unchanged
        hidden_size = self.gpt2.config.hidden_size
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(128, 2)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True
        )

        # Keep the original last-token pooling
        hidden_states = outputs.last_hidden_state
        last_positions = attention_mask.sum(dim=1) - 1

        pooled_output = hidden_states[
            torch.arange(
                hidden_states.size(0),
                device=hidden_states.device
            ),
            last_positions
        ]

        return self.classifier(pooled_output)


# Initialise the model
model = HPVNetGPT2Text().to(device)

# Show trainable LoRA parameters in the GPT-2 backbone
model.gpt2.print_trainable_parameters()

# Keep the original HECKTOR class weights
class_counts = np.bincount(y_train, minlength=2)
class_weights = torch.tensor(
    len(y_train) / (2 * np.maximum(class_counts, 1)),
    dtype=torch.float32,
    device=device
)

print(
    "Class weights [negative, positive]:",
    class_weights.detach().cpu().numpy()
)

print(
    "Total trainable parameters (LoRA + classifier):",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)


# Keep the original weighted loss
def weighted_loss(logits, labels):
    losses = nn.functional.cross_entropy(
        logits, labels, reduction="none"
    )
    return (losses * class_weights[labels]).mean()


# Optimise only LoRA adapters and the classification head
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("LoRA-enabled HECKTOR model initialised.")

# 10. VALIDATION SCORING AND THRESHOLD SELECTION
def get_scores(loader):
    model.eval()
    scores, labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids, mask = batch["input_ids"].to(device), batch["attention_mask"].to(device)
            logits = model(ids, mask)
            scores.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
            labels.extend(batch["label"].numpy())
    return np.asarray(scores), np.asarray(labels, dtype=int)

def evaluate_loss(loader):
    model.eval()
    total, n_total = 0.0, 0
    with torch.no_grad():
        for b in loader:
            ids, mask = b["input_ids"].to(device), b["attention_mask"].to(device)
            labels = b["label"].to(device)
            loss = weighted_loss(model(ids, mask), labels)
            n = len(labels); total += loss.item() * n; n_total += n
    return total / max(n_total, 1)

def choose_threshold(scores, labels):
    thresholds = np.unique(np.r_[0.0, scores, 1.0])
    values = [(float(t), balanced_accuracy_score(labels, scores >= t)) for t in thresholds]
    return max(values, key=lambda x: (x[1], -abs(x[0] - 0.5)))

# 11. TRAIN MODEL AND SAVE BEST VALIDATION CHECKPOINTS
OUT_DIR = Path("ROC_HECKTOR_GPT2_CLASSIFIER")
OUT_DIR.mkdir(parents=True, exist_ok=True)
paths = {
    "Validation_Loss": OUT_DIR / "best_model_validation_loss.pth",
    "Validation_ROC_AUC": OUT_DIR / "best_model_validation_roc_auc.pth",
    "Validation_PR_AUC": OUT_DIR / "best_model_validation_pr_auc.pth",
    "Validation_Balanced_Accuracy": OUT_DIR / "best_model_validation_balanced_accuracy.pth"
}
best = {
    "Validation_Loss": float("inf"), "Validation_ROC_AUC": -float("inf"),
    "Validation_PR_AUC": -float("inf"), "Validation_Balanced_Accuracy": -float("inf")
}
best_info, history = {}, []

for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum, n_sum = 0.0, 0
    for b in train_loader:
        ids, mask = b["input_ids"].to(device), b["attention_mask"].to(device)
        labels = b["label"].to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = weighted_loss(model(ids, mask), labels)
        loss.backward()
        optimizer.step()
        n = len(labels); loss_sum += loss.item() * n; n_sum += n

    train_loss = loss_sum / max(n_sum, 1)
    val_loss = evaluate_loss(val_loader)
    val_scores, val_labels = get_scores(val_loader)
    val_auc = roc_auc_score(val_labels, val_scores)
    val_ap = average_precision_score(val_labels, val_scores)
    threshold, val_ba = choose_threshold(val_scores, val_labels)
    metrics = {
        "Validation_Loss": val_loss, "Validation_ROC_AUC": val_auc,
        "Validation_PR_AUC": val_ap, "Validation_Balanced_Accuracy": val_ba
    }
    history.append({
        "Epoch": epoch, "Train_Loss": train_loss, **metrics,
        "Validation_Threshold": threshold
    })

    for name, value in metrics.items():
        improved = value < best[name] if name == "Validation_Loss" else value > best[name]
        if improved:
            best[name] = float(value)
            best_info[name] = {"epoch": epoch, "value": float(value), "threshold": threshold}
            torch.save(model.state_dict(), paths[name])

    if epoch % 50 == 0:
        print(f"Epoch {epoch}/{EPOCHS} | Train Loss={train_loss:.4f} | "
              f"Val Loss={val_loss:.4f} | ROC-AUC={val_auc:.4f} | "
              f"PR-AUC={val_ap:.4f} | BA={val_ba:.4f} | Threshold={threshold:.4f}")

pd.DataFrame(history).to_csv(OUT_DIR / "training_history.csv", index=False)
pd.DataFrame([
    {"Checkpoint": k, **v, "Path": str(paths[k])} for k, v in best_info.items()
]).to_csv(OUT_DIR / "best_validation_checkpoints.csv", index=False)

# 12. EVALUATE ALL CHECKPOINTS ON HELD-OUT TEST SET
results, predictions_by_model = [], {}
for name, path in paths.items():
    model.load_state_dict(torch.load(path, map_location=device))
    scores, labels = get_scores(test_loader)
    threshold = best_info[name]["threshold"]
    preds = (scores >= threshold).astype(int)
    auc, ap = roc_auc_score(labels, scores), average_precision_score(labels, scores)
    ba = balanced_accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, zero_division=0)
    cm = confusion_matrix(labels, preds, labels=[0, 1])

    print(f"\n=== {name} | Best epoch {best_info[name]['epoch']} | Threshold {threshold:.4f} ===")
    print(classification_report(
        labels, preds, labels=[0, 1],
        target_names=["HPV Negative", "HPV Positive"], digits=4, zero_division=0
    ))
    print("Confusion matrix [[TN, FP], [FN, TP]]:\n", cm)
    print(f"Test ROC-AUC={auc:.4f} | PR-AUC={ap:.4f} | BA={ba:.4f} | F1={f1:.4f}")

    results.append({
        "Dataset": "HECKTOR", "Model": "GPT-2 Classifier with LoRA",
        "Checkpoint_Selected_By": name, "Best_Epoch": best_info[name]["epoch"],
        "Validation_Threshold": threshold, "Test_ROC_AUC": auc,
        "Test_PR_AUC": ap, "Test_Balanced_Accuracy": ba, "Test_F1": f1,
        "TN": cm[0, 0], "FP": cm[0, 1], "FN": cm[1, 0], "TP": cm[1, 1]
    })
    predictions_by_model[name] = (labels, scores, preds, threshold)
    pd.DataFrame({
        "True_Label": labels, "HPV_Positive_Score": scores,
        "Predicted_Label": preds, "Threshold": threshold
    }).to_csv(OUT_DIR / f"test_predictions_{name.lower()}.csv", index=False)

results_df = pd.DataFrame(results)
results_df.to_csv(OUT_DIR / "test_metrics_by_checkpoint.csv", index=False)

# 13. SAVE ROC AND PRECISION-RECALL CURVES
for name, (labels, scores, preds, threshold) in predictions_by_model.items():
    safe = name.lower()
    fpr, tpr, roc_thresholds = roc_curve(labels, scores)
    precision, recall, _ = precision_recall_curve(labels, scores)
    auc, ap = roc_auc_score(labels, scores), average_precision_score(labels, scores)

    pd.DataFrame({"FPR": fpr, "TPR": tpr, "Threshold": roc_thresholds}).to_csv(
        OUT_DIR / f"roc_coordinates_{safe}.csv", index=False
    )
    pd.DataFrame({"Precision": precision, "Recall": recall}).to_csv(
        OUT_DIR / f"pr_coordinates_{safe}.csv", index=False
    )

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
    plt.plot([0, 1], [0, 1], "--")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"HECKTOR ROC — {name}"); plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"roc_{safe}.png", dpi=300, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"AP={ap:.3f}")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(f"HECKTOR Precision-Recall — {name}"); plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"pr_{safe}.png", dpi=300, bbox_inches="tight")
    plt.close()

# 14. SAVE SETTINGS AND COMPARISON RESULTS
settings = {
    "Dataset": "HECKTOR", "Model": MODEL_NAME, "Target": "HPV Status",
    "Features": ", ".join(FEATURES), "Number_of_Features": len(FEATURES),
    "Split": "65/15/20 stratified", "Seed": SEED, "Epochs": EPOCHS,
    "Batch_Size": BATCH_SIZE, "Learning_Rate": LEARNING_RATE,
    "Weight_Decay": WEIGHT_DECAY, "Dropout": DROPOUT,
    "Max_Length": MAX_LENGTH, "Optimizer": "AdamW",
    "Class_Weighted_Loss": True, "GPT2_Backbone_Frozen": True,
    "Model_Type": "GPT-2 Classifier with LoRA",
    "LoRA_Rank": 6,
    "LoRA_Alpha": 12,
    "LoRA_Target_Modules": "c_attn",
    "LoRA_Dropout": 0.0,
    "Checkpoint_Selection": "Validation loss, ROC-AUC, PR-AUC, balanced accuracy",
    "Threshold_Selection": "Maximum validation balanced accuracy"
}
pd.DataFrame([settings]).to_csv(OUT_DIR / "settings.csv", index=False)

comparison_file = Path("model_roc_pr_auc_comparison.csv")
comparison = results_df[[
    "Dataset", "Model", "Checkpoint_Selected_By", "Best_Epoch",
    "Test_ROC_AUC", "Test_PR_AUC", "Test_Balanced_Accuracy", "Test_F1"
]]
if comparison_file.exists():
    old = pd.read_csv(comparison_file)
    combined = pd.concat([old, comparison], ignore_index=True, sort=False)
    combined = combined.drop_duplicates(
        subset=["Dataset", "Model", "Checkpoint_Selected_By"], keep="last"
    )
else:
    combined = comparison
combined.to_csv(comparison_file, index=False)

print("\nAll outputs saved to:", OUT_DIR.resolve())
print("Comparison CSV:", comparison_file.resolve())

PyTorch: 2.14.1+cu130
Transformers: 5.22.1
PEFT: 0.21.2
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Device: cuda


Downloading...
From: https://drive.google.com/uc?id=1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv
To: /content/HPV2025.xlsx
100%|██████████| 66.0k/66.0k [00:00<00:00, 25.9MB/s]


Dataset: (588, 9) | Invalid/missing target rows dropped: 138
HPV classes (0=negative, 1=positive):
 HPV_Label
0     58
1    530
Name: count, dtype: int64
Train: n=381, class counts=[ 37 344]
Validation: n=89, class counts=[ 9 80]
Test: n=118, class counts=[ 12 106]


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

... (6 outputs left out)

/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2760: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 221,184 || all params: 124,660,992 || trainable%: 0.1774
Class weights [negative, positive]: [5.1486487  0.55377907]
Total trainable parameters (LoRA + classifier): 321410
LoRA-enabled HECKTOR model initialised.


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:910.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 50/500 | Train Loss=0.5195 | Val Loss=0.6997 | ROC-AUC=0.7944 | PR-AUC=0.9744 | BA=0.7875 | Threshold=0.8560
Epoch 100/500 | Train Loss=0.3749 | Val Loss=0.5238 | ROC-AUC=0.7972 | PR-AUC=0.9732 | BA=0.7764 | Threshold=0.5490
Epoch 150/500 | Train Loss=0.4101 | Val Loss=0.5660 | ROC-AUC=0.8000 | PR-AUC=0.9728 | BA=0.7764 | Threshold=0.6037
Epoch 200/500 | Train Loss=0.3471 | Val Loss=0.6023 | ROC-AUC=0.7347 | PR-AUC=0.9638 | BA=0.7063 | Threshold=0.8361
Epoch 250/500 | Train Loss=0.3188 | Val Loss=0.6201 | ROC-AUC=0.7653 | PR-AUC=0.9690 | BA=0.7438 | Threshold=0.9555
Epoch 300/500 | Train Loss=0.3048 | Val Loss=0.8158 | ROC-AUC=0.7306 | PR-AUC=0.9617 | BA=0.7063 | Threshold=0.9687
Epoch 350/500 | Train Loss=0.2906 | Val Loss=1.0841 | ROC-AUC=0.6778 | PR-AUC=0.9534 | BA=0.6632 | Threshold=0.9654
Epoch 400/500 | Train Loss=0.2183 | Val Loss=1.7748 | ROC-AUC=0.6361 | PR-AUC=0.9402 | BA=0.6771 | Threshold=0.8787
Epoch 450/500 | Train Loss=0.1449 | Val Loss=1.1648 | ROC-AUC=0.7694 | PR

In [2]:

# ============================================================
# SHAP + LIME EXPLANATIONS FOR ONE HPV-POSITIVE AND ONE HPV-NEGATIVE PATIENT
# HECKTOR GPT-2 LoRA notebook
# Creates 4 same-size plots + CSV + ZIP download
# ============================================================

!pip -q install shap lime

import os
import zipfile
import numpy as np
import pandas as pd
import torch
import shap
import matplotlib.pyplot as plt

from lime.lime_tabular import LimeTabularExplainer
from pathlib import Path
from IPython.display import display
from google.colab import files

# ------------------------------------------------------------
# 1. CHECK NOTEBOOK VARIABLES
# ------------------------------------------------------------

required_variables = [
    "model", "device", "tokenizer", "MAX_LENGTH",
    "train_df", "test_df", "y_test", "patient_to_text",
    "FEATURES", "OUT_DIR"
]

missing = [v for v in required_variables if v not in globals()]
if missing:
    raise NameError(
        f"Missing notebook variables: {missing}. "
        "Run the training and data preparation cells first."
    )

FEATURES = list(FEATURES)
assert len(FEATURES) == 8, (
    f"Expected 8 clinical features, but found {len(FEATURES)}."
)

# ------------------------------------------------------------
# 2. LOAD THE BEST VALIDATION ROC-AUC CHECKPOINT
# ------------------------------------------------------------

checkpoint_path = Path(OUT_DIR) / "best_model_validation_roc_auc.pth"

if not checkpoint_path.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {checkpoint_path}\n"
        "Make sure model training has completed."
    )

state = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state)
model.to(device)
model.eval()

print("Loaded checkpoint:", checkpoint_path)

# ------------------------------------------------------------
# 3. SELECT ONE ACTUAL POSITIVE AND ONE ACTUAL NEGATIVE
#    FROM THE HELD-OUT TEST SET
# ------------------------------------------------------------

y_test_array = np.asarray(y_test).astype(int)

positive_positions = np.where(y_test_array == 1)[0]
negative_positions = np.where(y_test_array == 0)[0]

if len(positive_positions) == 0 or len(negative_positions) == 0:
    raise ValueError(
        "The test set must contain at least one positive "
        "and one negative patient."
    )

positive_idx = int(positive_positions[0])
negative_idx = int(negative_positions[0])

positive_patient = test_df.iloc[positive_idx][FEATURES].copy()
negative_patient = test_df.iloc[negative_idx][FEATURES].copy()

print("\nSelected patients:")
print("Positive patient — actual label:", y_test_array[positive_idx])
print("Negative patient — actual label:", y_test_array[negative_idx])

# ------------------------------------------------------------
# 4. ENCODE THE EIGHT ORIGINAL CLINICAL FEATURES
#    Categorical features are mapped to integer codes.
#    Age remains numeric.
# ------------------------------------------------------------

numeric_features = [
    col for col in FEATURES
    if pd.api.types.is_numeric_dtype(train_df[col])
]

categorical_features = [
    col for col in FEATURES
    if col not in numeric_features
]

feature_to_idx = {name: i for i, name in enumerate(FEATURES)}

category_maps = {}
category_names = {}

for col in categorical_features:
    values = (
        train_df[col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )

    categories = sorted(values.unique().tolist())

    if "Unknown" not in categories:
        categories.append("Unknown")

    category_maps[col] = {
        value: i for i, value in enumerate(categories)
    }
    category_names[feature_to_idx[col]] = categories


def encode_dataframe(dataframe):
    """Convert clinical rows to the numeric representation used by SHAP/LIME."""
    result = pd.DataFrame(index=dataframe.index)

    for col in FEATURES:
        if col in numeric_features:
            result[col] = pd.to_numeric(
                dataframe[col], errors="coerce"
            ).fillna(
                pd.to_numeric(train_df[col], errors="coerce").median()
            )
        else:
            values = dataframe[col].fillna("Unknown").astype(str).str.strip()
            mapping = category_maps[col]
            unknown_code = mapping["Unknown"]

            result[col] = values.map(mapping).fillna(
                unknown_code
            ).astype(float)

    return result[FEATURES].astype(float)


def decode_dataframe(matrix):
    """Convert perturbed numeric rows back to clinical feature values."""
    matrix = np.asarray(matrix)
    decoded = pd.DataFrame(columns=FEATURES)

    for j, col in enumerate(FEATURES):
        if col in numeric_features:
            decoded[col] = matrix[:, j].astype(float)
        else:
            categories = list(category_maps[col].keys())
            codes = np.rint(matrix[:, j]).astype(int)
            codes = np.clip(codes, 0, len(categories) - 1)

            decoded[col] = [categories[c] for c in codes]

    return decoded[FEATURES]


X_train = encode_dataframe(train_df[FEATURES])
X_test = encode_dataframe(test_df[FEATURES])

positive_x = X_test.iloc[[positive_idx]].to_numpy()
negative_x = X_test.iloc[[negative_idx]].to_numpy()

# Background sample for SHAP
background_size = min(20, len(X_train))
background = X_train.sample(
    n=background_size,
    random_state=42
).to_numpy()

# ------------------------------------------------------------
# 5. GPT-2 LoRA PREDICTION FUNCTION
# ------------------------------------------------------------

def predict_probabilities(matrix, batch_size=16):
    """
    Convert perturbed clinical features into the notebook's
    narrative prompts and return [P(negative), P(positive)].
    """
    decoded = decode_dataframe(matrix)
    texts = [
        patient_to_text(row)
        for _, row in decoded.iterrows()
    ]

    all_probabilities = []

    model.eval()

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        with torch.no_grad():
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            probabilities = torch.softmax(logits, dim=1)

        all_probabilities.append(
            probabilities.detach().cpu().numpy()
        )

    return np.vstack(all_probabilities)


# Confirm predictions for the two selected patients
positive_probability = float(
    predict_probabilities(positive_x)[0, 1]
)
negative_probability = float(
    predict_probabilities(negative_x)[0, 1]
)

print(f"\nPositive patient's predicted HPV-positive probability: "
      f"{positive_probability:.4f}")

print(f"Negative patient's predicted HPV-positive probability: "
      f"{negative_probability:.4f}")

# ------------------------------------------------------------
# 6. OUTPUT DIRECTORY AND PLOT SETTINGS
# ------------------------------------------------------------

EXPLAIN_DIR = Path(OUT_DIR) / "SHAP_LIME_Individual_Patients"
EXPLAIN_DIR.mkdir(parents=True, exist_ok=True)

FIGSIZE = (8, 4.8)
DPI = 300

def save_standard_plot(path):
    """Apply identical dimensions and resolution to every plot."""
    fig = plt.gcf()
    fig.set_size_inches(*FIGSIZE)
    fig.savefig(
        path,
        dpi=DPI,
        bbox_inches=None,
        facecolor="white"
    )
    plt.close(fig)


# ------------------------------------------------------------
# 7. SHAP WATERFALL PLOTS
#    Each plot explains the probability of the patient's
#    actual class: positive for the positive patient,
#    negative for the negative patient.
# ------------------------------------------------------------

def shap_class_probability(matrix, class_number):
    return predict_probabilities(matrix)[:, class_number]


def create_shap_waterfall(patient_x, actual_class, patient_name):
    print(f"\nCalculating SHAP for {patient_name} patient...")

    explainer = shap.KernelExplainer(
        lambda data: shap_class_probability(data, actual_class),
        background
    )

    shap_values = explainer.shap_values(
        patient_x,
        nsamples=300
    )

    shap_values = np.asarray(shap_values)

    if shap_values.ndim > 1:
        shap_values = shap_values.reshape(-1, len(FEATURES))[0]

    base_value = np.asarray(explainer.expected_value).squeeze()
    if base_value.ndim > 0:
        base_value = float(base_value.reshape(-1)[0])
    else:
        base_value = float(base_value)

    # Original clinical values for display
    raw_values = (
        positive_patient if actual_class == 1 else negative_patient
    )
    display_values = np.array(
        [str(raw_values[col]) for col in FEATURES],
        dtype=object
    )

    explanation = shap.Explanation(
        values=shap_values,
        base_values=base_value,
        data=patient_x[0],
        display_data=display_values,
        feature_names=FEATURES
    )

    shap.plots.waterfall(
        explanation,
        max_display=8,
        show=False
    )

    plt.title(
        f"{patient_name} Patient — SHAP Waterfall",
        fontsize=12
    )
    plt.tight_layout()

    output_path = (
        EXPLAIN_DIR /
        f"HPV_{patient_name}_SHAP_waterfall.png"
    )

    save_standard_plot(output_path)
    print("Saved:", output_path)


create_shap_waterfall(
    positive_x, actual_class=1, patient_name="positive"
)

create_shap_waterfall(
    negative_x, actual_class=0, patient_name="negative"
)

# ------------------------------------------------------------
# 8. LIME EXPLAINER
# ------------------------------------------------------------

lime_explainer = LimeTabularExplainer(
    training_data=X_train.to_numpy(),
    feature_names=FEATURES,
    class_names=["HPV Negative", "HPV Positive"],
    categorical_features=[
        feature_to_idx[col] for col in categorical_features
    ],
    categorical_names=category_names,
    mode="classification",
    discretize_continuous=True,
    random_state=42
)


def create_lime_plot(patient_x, actual_class, patient_name):
    print(f"\nCalculating LIME for {patient_name} patient...")

    explanation = lime_explainer.explain_instance(
        data_row=patient_x[0],
        predict_fn=predict_probabilities,
        labels=(actual_class,),
        num_features=8,
        num_samples=1500
    )

    explanation_list = explanation.as_list(
        label=actual_class
    )

    # Sort by absolute contribution for readability
    explanation_list = sorted(
        explanation_list,
        key=lambda item: abs(item[1])
    )

    labels = [item[0] for item in explanation_list]
    values = [item[1] for item in explanation_list]

    plt.figure(figsize=FIGSIZE)

    bar_colors = [
        "seagreen" if value >= 0 else "indianred"
        for value in values
    ]

    plt.barh(
        labels,
        values,
        color=bar_colors
    )

    plt.axvline(
        0,
        color="black",
        linewidth=0.8
    )

    plt.xlabel(
        f"Contribution to {('HPV Positive' if actual_class == 1 else 'HPV Negative')} class"
    )
    plt.title(
        f"{patient_name.capitalize()} Patient — LIME Explanation",
        fontsize=12
    )
    plt.tight_layout()

    output_path = (
        EXPLAIN_DIR /
        f"HPV_{patient_name}_LIME_bar.png"
    )

    save_standard_plot(output_path)
    print("Saved:", output_path)


create_lime_plot(
    positive_x, actual_class=1, patient_name="positive"
)

create_lime_plot(
    negative_x, actual_class=0, patient_name="negative"
)

# ------------------------------------------------------------
# 9. SAVE PATIENT SUMMARY
# ------------------------------------------------------------

summary = pd.DataFrame([
    {
        "Patient": "HPV Positive",
        "Actual_Label": 1,
        "Predicted_HP V_Positive_Probability": positive_probability
    },
    {
        "Patient": "HPV Negative",
        "Actual_Label": 0,
        "Predicted_HP V_Positive_Probability": negative_probability
    }
])

# Use clean column names
summary.columns = [
    "Patient",
    "Actual_Label",
    "Predicted_HPV_Positive_Probability"
]

summary_path = EXPLAIN_DIR / "patient_prediction_summary.csv"
summary.to_csv(summary_path, index=False)

# ------------------------------------------------------------
# 10. VERIFY THAT ALL FOUR PLOTS HAVE THE SAME DIMENSIONS
# ------------------------------------------------------------

from PIL import Image

plot_files = [
    EXPLAIN_DIR / "HPV_positive_SHAP_waterfall.png",
    EXPLAIN_DIR / "HPV_positive_LIME_bar.png",
    EXPLAIN_DIR / "HPV_negative_SHAP_waterfall.png",
    EXPLAIN_DIR / "HPV_negative_LIME_bar.png"
]

print("\nPLOT DIMENSION CHECK")
for path in plot_files:
    if not path.exists():
        raise FileNotFoundError(f"Expected plot was not created: {path}")

    with Image.open(path) as img:
        print(f"{path.name}: {img.size} pixels")

# ------------------------------------------------------------
# 11. CREATE ZIP FILE AND DOWNLOAD AUTOMATICALLY
# ------------------------------------------------------------

zip_path = Path(OUT_DIR) / "HECKTOR_GPT2_LoRA_SHAP_LIME_4_PLOTS.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for path in plot_files:
        zip_file.write(path, arcname=path.name)

    zip_file.write(
        summary_path,
        arcname=summary_path.name
    )

print("\nAll four plots and the patient summary are saved.")
print("ZIP file:", zip_path)

files.download(str(zip_path))


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Loaded checkpoint: ROC_HECKTOR_GPT2_CLASSIFIER/best_model_validation_roc_auc.pth

Selected patients:
Positive patient — actual label: 1
Negative patient — actual label: 0

Positive patient's predicted HPV-positive probability: 0.4586
Negative patient's predicted HPV-positive probability: 0.4552

Calculating SHAP for positive patient...


  0%|          | 0/1 [00:00<?, ?it/s]

Saved: ROC_HECKTOR_GPT2_CLASSIFIER/SHAP_LIME_Individual_Patients/HPV_positive_SHAP_waterfall.png

Calculating SHAP for negative patient...


  0%|          | 0/1 [00:00<?, ?it/s]

Saved: ROC_HECKTOR_GPT2_CLASSIFIER/SHAP_LIME_Individual_Patients/HPV_negative_SHAP_waterfall.png

Calculating LIME for positive patient...
Saved: ROC_HECKTOR_GPT2_CLASSIFIER/SHAP_LIME_Individual_Patients/HPV_positive_LIME_bar.png

Calculating LIME for negative patient...
Saved: ROC_HECKTOR_GPT2_CLASSIFIER/SHAP_LIME_Individual_Patients/HPV_negative_LIME_bar.png

PLOT DIMENSION CHECK
HPV_positive_SHAP_waterfall.png: (2400, 1440) pixels
HPV_positive_LIME_bar.png: (2400, 1440) pixels
HPV_negative_SHAP_waterfall.png: (2400, 1440) pixels
HPV_negative_LIME_bar.png: (2400, 1440) pixels

All four plots and the patient summary are saved.
ZIP file: ROC_HECKTOR_GPT2_CLASSIFIER/HECKTOR_GPT2_LoRA_SHAP_LIME_4_PLOTS.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>